In [ ]:
# Install Necessary Libraries
# ! pip install -q opencv-python ultralytics lap

In [10]:
import cv2
from IPython.display import display, Video
from ultralytics import YOLO

# Load the YOLOv8 model
model = YOLO("yolov8n.pt")  # Replace with your model file if needed

# Upload the video file or use a URL to load the video
# video_path = "./car.mp4"  # Change this path as needed
video_path = "./people.mp4"  # Change this path as needed
cap = cv2.VideoCapture(video_path)

# --- NEW: Initialize a set to store unique tracking IDs ---
unique_ids = set()

# Get the width, height, and frames per second of the video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define the output video file
output_path = './people_annotated.mp4'
# output_path = './car_annotated.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for output video
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# track_object_type = 2 # Car
track_object_type = 0 # People

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLOv8 tracking on the frame, persisting tracks between frames
        results = model.track(frame, conf=0.5, verbose=False,persist=True,classes=[track_object_type])

        # --- NEW: Logic to extract and store unique IDs ---
        if results[0].boxes.id is not None:
            # Get the IDs from the current frame
            # .cpu().numpy() moves the data from GPU to CPU for easy processing
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            
            # Add these IDs to our set (sets only keep unique values)
            for obj_id in track_ids:
                unique_ids.add(obj_id)

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        # Write the annotated frame to the output video file
        out.write(annotated_frame)
    else:
        break

# Release the video capture and writer objects
cap.release()
out.release()

print("-" * 30)
print("TRACKING COMPLETE")
print("-" * 30)
print(f"Total unique objects detected: {len(unique_ids)}")
print(f"List of IDs tracked: {sorted(list(unique_ids))}")
print("-" * 30)

# Display the annotated video in Google Colab
display(Video(output_path))

------------------------------
TRACKING COMPLETE
------------------------------
Total unique objects detected: 54
List of IDs tracked: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(12), np.int64(14), np.int64(16), np.int64(17), np.int64(20), np.int64(21), np.int64(24), np.int64(25), np.int64(28), np.int64(29), np.int64(30), np.int64(34), np.int64(35), np.int64(36), np.int64(38), np.int64(41), np.int64(42), np.int64(44), np.int64(45), np.int64(48), np.int64(50), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(60), np.int64(61), np.int64(65), np.int64(66), np.int64(67), np.int64(69), np.int64(70), np.int64(71), np.int64(73), np.int64(74), np.int64(75), np.int64(76), np.int64(81), np.int64(83), np.int64(84), np.int64(85), np.int64(86), np.int64(87), np.int64(88)]
------------------------------
